# ♟️ Chess Steganography Detector — ML Training Pipeline

---

## Overview

This notebook trains a **binary classifier** to detect whether a chess game (PGN format) contains hidden steganographic data.

### How the steganography works
The `stego_engine.py` tool hides arbitrary binary payloads inside chess games by encoding each bit as a **move choice**:
- Bit `0` → pick `legal_moves[0]` (alphabetically first UCI move)
- Bit `1` → pick `legal_moves[1]` (alphabetically second UCI move)

This creates a detectable statistical pattern: stego games **always** pick moves ranked 0 or 1, while normal human games spread across all available moves.

### Dataset
| Class | Label | Source | Count |
|-------|-------|--------|-------|
| Normal games | 0 | Synthetic random / Lichess 500-600 Elo | 500 |
| Stego games  | 1 | Generated by `stego_engine.py` | 500 |

### Pipeline
1. Load & Explore → 2. Feature Analysis → 3. Train 3 Models → 4. Evaluate → 5. Save → 6. Inference Demo

In [ ]:
# ── Cell 2: Install & Imports ─────────────────────────────────────────────────
# All libraries are pre-installed in Colab. Uncomment if running locally.
# !pip install pandas numpy scikit-learn matplotlib seaborn joblib -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib
import warnings
import os

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc, classification_report
)

warnings.filterwarnings('ignore')

# ── Plot style ────────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-darkgrid')
PALETTE  = {'Normal (0)': '#4ECDC4', 'Stego (1)': '#FF6B6B'}
COLOR_N  = '#4ECDC4'   # teal  — normal
COLOR_S  = '#FF6B6B'   # coral — stego
FIGSIZE  = (14, 5)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Libraries loaded successfully.')
print(f'pandas  {pd.__version__} | numpy {np.__version__} | sklearn {__import__("sklearn").__version__}')

In [ ]:
# ── Cell 3: Load & Explore Data ───────────────────────────────────────────────

CSV_PATH = 'dataset.csv'   # adjust path if needed in Colab

df = pd.read_csv(CSV_PATH)

FEATURE_COLS = ['avg_move_rank', 'rank_0_ratio', 'rank_1_ratio',
                'rank_0_or_1_ratio', 'rank_variance', 'game_length']
TARGET_COL   = 'label'

X = df[FEATURE_COLS].values
y = df[TARGET_COL].values

# ── Basic info ────────────────────────────────────────────────────────────────
print('=' * 52)
print('  Dataset Overview')
print('=' * 52)
print(f'  Shape          : {df.shape}')
print(f'  Features       : {FEATURE_COLS}')
print(f'  Label 0 (normal): {(y == 0).sum()} samples')
print(f'  Label 1 (stego) : {(y == 1).sum()} samples')
print()
print(df.describe().round(4))

# ── Visualizations ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)
fig.suptitle('Dataset Exploration', fontsize=15, fontweight='bold', y=1.01)

# 1. Class distribution bar chart
ax = axes[0]
counts = pd.Series(y).value_counts().sort_index()
bars = ax.bar(['Normal (0)', 'Stego (1)'], counts.values,
              color=[COLOR_N, COLOR_S], edgecolor='white', linewidth=1.2, width=0.5)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(val), ha='center', va='bottom', fontweight='bold', fontsize=12)
ax.set_title('Class Distribution', fontsize=13, fontweight='bold')
ax.set_ylabel('Sample Count')
ax.set_ylim(0, counts.max() * 1.15)
ax.spines[['top','right']].set_visible(False)

# 2. Feature correlation heatmap
ax = axes[1]
corr = df[FEATURE_COLS + [TARGET_COL]].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax,
            annot_kws={'size': 8}, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.setp(ax.get_xticklabels(), rotation=30, ha='right', fontsize=8)
plt.setp(ax.get_yticklabels(), rotation=0, fontsize=8)

plt.tight_layout()
plt.savefig('plot_exploration.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plot_exploration.png')

In [ ]:
# ── Cell 4: Feature Analysis — Normal vs Stego Boxplots ──────────────────────

df_plot = df.copy()
df_plot['Class'] = df_plot[TARGET_COL].map({0: 'Normal (0)', 1: 'Stego (1)'})

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Feature Distributions: Normal vs Stego Games',
             fontsize=16, fontweight='bold', y=1.01)
axes = axes.flatten()

# Feature descriptions for richer labels
FEAT_DESC = {
    'avg_move_rank'    : 'Avg Move Rank\n(lower = more stego-like)',
    'rank_0_ratio'     : 'Rank-0 Move Ratio\n(% of 1st moves chosen)',
    'rank_1_ratio'     : 'Rank-1 Move Ratio\n(% of 2nd moves chosen)',
    'rank_0_or_1_ratio': 'Rank-0-or-1 Ratio\n(key stego signal)',
    'rank_variance'    : 'Rank Variance\n(spread of move choices)',
    'game_length'      : 'Game Length\n(total moves)',
}

for ax, feat in zip(axes, FEATURE_COLS):
    sns.boxplot(
        data=df_plot, x='Class', y=feat,
        palette={'Normal (0)': COLOR_N, 'Stego (1)': COLOR_S},
        width=0.45, linewidth=1.2, flierprops=dict(marker='o', markersize=3, alpha=0.5),
        ax=ax
    )
    # Overlay individual points with jitter
    sns.stripplot(
        data=df_plot, x='Class', y=feat,
        palette={'Normal (0)': COLOR_N, 'Stego (1)': COLOR_S},
        size=2.5, alpha=0.25, jitter=True, ax=ax
    )
    ax.set_title(FEAT_DESC[feat], fontsize=10, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel(feat, fontsize=8)
    ax.spines[['top','right']].set_visible(False)

    # Annotate mean difference
    mean_n = df_plot.loc[df_plot[TARGET_COL]==0, feat].mean()
    mean_s = df_plot.loc[df_plot[TARGET_COL]==1, feat].mean()
    ratio  = abs(mean_n - mean_s) / (abs(mean_n) + 1e-9)
    ax.set_xlabel(f'Mean diff: {mean_n:.2f} vs {mean_s:.2f}  (ratio {ratio:.1f}x)',
                  fontsize=7.5, color='gray')

plt.tight_layout()
plt.savefig('plot_features.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plot_features.png')

# ── Separability summary ──────────────────────────────────────────────────────
print('\nSeparability Summary (mean per class):')
print(f"{'Feature':<22} {'Normal':>10} {'Stego':>10} {'Ratio':>8}")
print('-' * 54)
for feat in FEATURE_COLS:
    mn = df.loc[df[TARGET_COL]==0, feat].mean()
    ms = df.loc[df[TARGET_COL]==1, feat].mean()
    r  = abs(mn-ms)/(abs(mn)+1e-9)
    print(f"{feat:<22} {mn:>10.3f} {ms:>10.3f} {r:>7.1f}x")

In [ ]:
# ── Cell 5: Train 3 Models & Compare ─────────────────────────────────────────

# ── Train/test split ──────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')

# ── Model definitions ─────────────────────────────────────────────────────────
# LogisticRegression benefits from scaling; RF and GB are scale-invariant.
models = {
    'RandomForest': Pipeline([
        ('clf', RandomForestClassifier(
            n_estimators=100,
            max_depth=None,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),
    'LogisticRegression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE
        ))
    ]),
    'GradientBoosting': Pipeline([
        ('clf', GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=4,
            random_state=RANDOM_STATE
        ))
    ]),
}

# ── 5-fold cross-validation + test set evaluation ─────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

results = []
trained_models = {}

for name, pipeline in models.items():
    print(f'\nTraining {name}...')

    # Cross-validation
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=cv,
                                scoring='accuracy', n_jobs=-1)

    # Final fit on full training set
    pipeline.fit(X_train, y_train)
    trained_models[name] = pipeline

    # Test set predictions
    y_pred = pipeline.predict(X_test)

    row = {
        'Model'       : name,
        'CV Accuracy' : f'{cv_scores.mean():.4f} ± {cv_scores.std():.4f}',
        'Test Acc'    : accuracy_score(y_test, y_pred),
        'Precision'   : precision_score(y_test, y_pred),
        'Recall'      : recall_score(y_test, y_pred),
        'F1'          : f1_score(y_test, y_pred),
    }
    results.append(row)
    print(f'  CV Acc: {row["CV Accuracy"]} | Test Acc: {row["Test Acc"]:.4f}')

# ── Comparison table ──────────────────────────────────────────────────────────
results_df = pd.DataFrame(results)
print('\n' + '=' * 72)
print('  Model Comparison Table')
print('=' * 72)
print(results_df.to_string(index=False))
print('=' * 72)

# ── Visual comparison bar chart ───────────────────────────────────────────────
metrics_to_plot = ['Test Acc', 'Precision', 'Recall', 'F1']
model_names     = results_df['Model'].tolist()
bar_colors      = ['#6C5CE7', '#00B894', '#FDCB6E']

x     = np.arange(len(metrics_to_plot))
width = 0.22

fig, ax = plt.subplots(figsize=(12, 5))
for i, (name, row) in enumerate(zip(model_names, results)):
    vals = [row[m] for m in metrics_to_plot]
    bars = ax.bar(x + i*width, vals, width, label=name,
                  color=bar_colors[i], edgecolor='white', linewidth=0.8)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xticks(x + width)
ax.set_xticklabels(metrics_to_plot, fontsize=11)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Model Performance Comparison (Test Set)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)
ax.axhline(1.0, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)

plt.tight_layout()
plt.savefig('plot_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plot_model_comparison.png')

# Identify best model by F1
best_row   = max(results, key=lambda r: r['F1'])
best_name  = best_row['Model']
best_model = trained_models[best_name]
print(f'\n>>> Best model by F1: {best_name} (F1 = {best_row["F1"]:.4f})')

In [ ]:
# ── Cell 6: Best Model Deep Dive ─────────────────────────────────────────────

y_pred      = best_model.predict(X_test)
y_prob      = best_model.predict_proba(X_test)[:, 1]  # prob of stego

print(f'Deep Dive: {best_name}')
print('=' * 52)
print(classification_report(y_test, y_pred,
      target_names=['Normal (0)', 'Stego (1)']))

fig = plt.figure(figsize=(18, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)
fig.suptitle(f'Best Model Deep Dive — {best_name}',
             fontsize=15, fontweight='bold')

# ── 1. Confusion Matrix ───────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0])
cm  = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Stego'],
            yticklabels=['Normal', 'Stego'],
            linewidths=1, linecolor='white',
            annot_kws={'size': 14, 'weight': 'bold'}, ax=ax1)
ax1.set_xlabel('Predicted', fontsize=11)
ax1.set_ylabel('Actual',    fontsize=11)
ax1.set_title('Confusion Matrix', fontsize=13, fontweight='bold')

# ── 2. ROC Curve ─────────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1])
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc     = auc(fpr, tpr)
ax2.plot(fpr, tpr, color='#6C5CE7', lw=2.5,
         label=f'AUC = {roc_auc:.4f}')
ax2.fill_between(fpr, tpr, alpha=0.12, color='#6C5CE7')
ax2.plot([0,1], [0,1], 'k--', lw=1, alpha=0.5, label='Random')
ax2.set_xlabel('False Positive Rate', fontsize=11)
ax2.set_ylabel('True Positive Rate',  fontsize=11)
ax2.set_title('ROC Curve', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11)
ax2.spines[['top','right']].set_visible(False)

# ── 3. Feature Importance (RandomForest / GradientBoosting) ──────────────────
ax3 = fig.add_subplot(gs[2])
clf = best_model.named_steps.get('clf')
if hasattr(clf, 'feature_importances_'):
    importances = clf.feature_importances_
    sorted_idx  = np.argsort(importances)[::-1]
    feat_sorted = [FEATURE_COLS[i] for i in sorted_idx]
    imp_sorted  = importances[sorted_idx]

    bar_colors_fi = plt.cm.viridis(np.linspace(0.2, 0.85, len(feat_sorted)))
    bars = ax3.barh(feat_sorted[::-1], imp_sorted[::-1],
                    color=bar_colors_fi[::-1], edgecolor='white')
    for bar, val in zip(bars, imp_sorted[::-1]):
        ax3.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=9)
    ax3.set_xlabel('Importance', fontsize=11)
    ax3.set_title('Feature Importance', fontsize=13, fontweight='bold')
    ax3.set_xlim(0, imp_sorted.max() * 1.18)
    ax3.spines[['top','right']].set_visible(False)
else:
    # LogisticRegression: use |coefficients|
    scaler = best_model.named_steps.get('scaler')
    coefs  = np.abs(clf.coef_[0])
    sorted_idx  = np.argsort(coefs)[::-1]
    feat_sorted = [FEATURE_COLS[i] for i in sorted_idx]
    coef_sorted = coefs[sorted_idx]
    bar_colors_fi = plt.cm.plasma(np.linspace(0.2, 0.85, len(feat_sorted)))
    ax3.barh(feat_sorted[::-1], coef_sorted[::-1], color=bar_colors_fi[::-1])
    ax3.set_xlabel('|Coefficient|', fontsize=11)
    ax3.set_title('LogReg |Coefficients|', fontsize=13, fontweight='bold')
    ax3.spines[['top','right']].set_visible(False)

plt.savefig('plot_deepdive.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plot_deepdive.png')
print(f'\nAUC-ROC: {roc_auc:.4f}')

In [ ]:
# ── Cell 7: Save Best Model ───────────────────────────────────────────────────

MODEL_PATH = 'stego_detector.pkl'

# Save model metadata alongside for reproducibility
model_bundle = {
    'model'        : best_model,
    'model_name'   : best_name,
    'feature_cols' : FEATURE_COLS,
    'metrics'      : best_row,
    'train_size'   : X_train.shape[0],
    'test_size'    : X_test.shape[0],
}

joblib.dump(model_bundle, MODEL_PATH, compress=3)

size_kb = os.path.getsize(MODEL_PATH) / 1024
print('=' * 48)
print('  Model saved successfully!')
print('=' * 48)
print(f'  File       : {MODEL_PATH}')
print(f'  Size       : {size_kb:.1f} KB')
print(f'  Model type : {best_name}')
print(f'  Test Acc   : {best_row["Test Acc"]:.4f}')
print(f'  F1 Score   : {best_row["F1"]:.4f}')
print(f'  AUC-ROC    : {roc_auc:.4f}')
print(f'  Features   : {FEATURE_COLS}')
print('=' * 48)

In [ ]:
# ── Cell 8: Quick Inference Demo ─────────────────────────────────────────────
#
# This shows how to load the saved model and use it on a single game.
# In production, you would:
#   1. Load a PGN file
#   2. Call feature_extractor.extract_features(game)
#   3. Pass the 6 features to predict_game() below

# ── Load saved bundle ─────────────────────────────────────────────────────────
bundle          = joblib.load('stego_detector.pkl')
loaded_model    = bundle['model']
loaded_features = bundle['feature_cols']
print(f'Loaded model: {bundle["model_name"]}')
print(f'Expected features: {loaded_features}')

# ── Inference function ────────────────────────────────────────────────────────
def predict_game(features: list, threshold: float = 0.5) -> dict:
    """
    Predict whether a chess game contains hidden steganographic data.

    Args:
        features  : list of 6 floats in order:
                    [avg_move_rank, rank_0_ratio, rank_1_ratio,
                     rank_0_or_1_ratio, rank_variance, game_length]
        threshold : decision boundary (default 0.5)

    Returns:
        dict with keys: prediction, confidence, label, verdict
    """
    X_input = np.array(features, dtype=float).reshape(1, -1)
    prob    = loaded_model.predict_proba(X_input)[0, 1]   # P(stego)
    label   = int(prob >= threshold)

    verdict = '*** STEGO DETECTED ***' if label == 1 else 'NORMAL GAME'

    return {
        'prediction' : label,
        'confidence' : round(float(prob), 4),
        'label_name' : 'Stego (1)' if label == 1 else 'Normal (0)',
        'verdict'    : verdict,
    }


# ── Demo: 3 example predictions ───────────────────────────────────────────────
examples = [
    {
        'name'    : 'Clearly STEGO (rank_0_or_1_ratio = 1.0)',
        'features': [0.46, 0.54, 0.46, 1.00, 0.25, 32.0],
    },
    {
        'name'    : 'Clearly NORMAL (high rank spread)',
        'features': [14.2, 0.04, 0.04, 0.08, 91.8, 40.0],
    },
    {
        'name'    : 'Ambiguous (moderate rank_0_or_1_ratio)',
        'features': [5.0, 0.20, 0.18, 0.38, 25.0, 30.0],
    },
]

print('\n' + '=' * 60)
print('  Inference Demo')
print('=' * 60)

for ex in examples:
    result = predict_game(ex['features'])
    print(f"\n  Game : {ex['name']}")
    print(f"  Input: {dict(zip(loaded_features, ex['features']))}")
    print(f"  >>> {result['verdict']}")
    print(f"      P(stego) = {result['confidence']:.4f} | Label = {result['label_name']}")

print('\n' + '=' * 60)

# ── Batch prediction on entire test set (sanity check) ────────────────────────
batch_preds = [predict_game(row) for row in X_test.tolist()]
batch_labels = [p['prediction'] for p in batch_preds]
batch_acc    = (np.array(batch_labels) == y_test).mean()
print(f'\nSanity check — batch accuracy on test set: {batch_acc:.4f}')
print('(Should match Cell 5 test accuracy)')